# Quantum Fourier Transform — Amazon Braket

The QFT maps a computational basis state $|j\rangle$ to a uniform
superposition with phases encoding $j$:

$$\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_k e^{2\pi i j k / N} |k\rangle$$

We build the QFT from Hadamard and controlled-phase gates on 3 qubits,
apply it to $|5\rangle = |101\rangle$, verify the amplitudes against the
analytical DFT formula, and then check the QFT–IQFT identity on all
eight basis states.

In [ ]:
import cmath

from braket.circuits import Circuit, ResultType
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()
N_QUBITS = 3
N_STATES = 2 ** N_QUBITS

## Helper functions

Braket uses big-endian state-vector indexing: qubit 0 is the MSB, so
index $= q_0 \cdot 2^{n-1} + q_1 \cdot 2^{n-2} + \cdots + q_{n-1} \cdot 2^0$.

In [ ]:
def prepare_basis(circuit, j, n=N_QUBITS):
    """Prepare |j> with Braket big-endian qubit ordering."""
    for i in range(n):
        if (j >> (n - 1 - i)) & 1:
            circuit.x(i)

def sv_probs(circuit):
    """Run circuit, return (statevector, probabilities)."""
    circuit.add_result_type(ResultType.StateVector())
    circuit.add_result_type(ResultType.Probability())
    result = device.run(circuit, shots=0).result()
    sv = [complex(a) for a in result.result_types[0].value]
    probs = [float(p) for p in result.result_types[1].value]
    return sv, probs

## QFT circuit builder

The QFT applies Hadamard and controlled-phase gates in the standard
textbook order, with a final SWAP to match Braket's big-endian
convention.  The resulting unitary equals the DFT matrix.

In [ ]:
def qft_circuit(n=N_QUBITS):
    circuit = Circuit()
    for i in range(n):
        circuit.h(i)
        for j in range(i + 1, n):
            k = j - i + 1
            angle = 2.0 * cmath.pi / (2 ** k)
            circuit.cphaseshift(j, i, angle)
    for i in range(n // 2):
        circuit.swap(i, n - 1 - i)
    return circuit

def iqft_circuit(n=N_QUBITS):
    """Inverse QFT: reverse gate order, negate phase angles."""
    circuit = Circuit()
    for i in range(n // 2):
        circuit.swap(i, n - 1 - i)
    for i in range(n - 1, -1, -1):
        for j in range(n - 1, i, -1):
            k = j - i + 1
            angle = -2.0 * cmath.pi / (2 ** k)
            circuit.cphaseshift(j, i, angle)
        circuit.h(i)
    return circuit

## Apply QFT to $|5\rangle = |101\rangle$

Expected: $\text{QFT}|5\rangle_k = e^{2\pi i \cdot 5k/8}/\sqrt{8}$ for each $k$.

In [ ]:
j = 5

circuit = Circuit()
prepare_basis(circuit, j)
circuit.add_circuit(qft_circuit())

sv, probs = sv_probs(circuit)

print(f"QFT on |{j}>")
print("k  | measured amp       | expected amp         | |amp|^2")
for k in range(N_STATES):
    bits_k = format(k, f'0{N_QUBITS}b')
    m = sv[k]
    e = cmath.exp(2j * cmath.pi * j * k / N_STATES) / N_STATES ** 0.5
    p = probs[k]
    match = "ok" if abs(m - e) < 1e-6 else "MISMATCH"
    print(f"|{bits_k}>  {m.real:+.4f}{m.imag:+.4f}i  "
          f"{e.real:+.4f}{e.imag:+.4f}i  {p:.4f}  {match}")

## QFT then IQFT = identity

Applying the QFT and its inverse should return each basis state unchanged.

In [ ]:
all_ok = True
for j in range(N_STATES):
    circuit = Circuit()
    prepare_basis(circuit, j)
    circuit.add_circuit(qft_circuit())
    circuit.add_circuit(iqft_circuit())

    sv, _ = sv_probs(circuit)
    out_idx = max(range(len(sv)), key=lambda i: abs(sv[i]))
    ok = out_idx == j
    all_ok = all_ok and ok
    print(f"  |{j}> -> |{out_idx}>  {'ok' if ok else 'FAIL'}")
print(f"\nAll passed: {all_ok}")